# Демо 3. SFT: від повного донавчання до LoRA і QLoRA

У цьому ноутбуці донавчаємо моделі **Qwen3** трьох розмірів виконувати одну вузьку прикладну задачу:
**класифікувати звернення клієнтів служби підтримки** й відповідати лише JSON із категорією та наміром звернення.

### Навіщо потрібен SFT

Інтернет-магазин щодня отримує тисячі повідомлень від клієнтів: скасувати замовлення, змінити адресу доставки,
повернути кошти тощо. Щоб автоматично передавати кожне звернення у відповідний відділ, програмі потрібна відповідь
у строго заданому форматі, наприклад `{"category": "ORDER", "intent": "cancel_order"}`, з мітками з переліку компанії.

Загальна instruct-модель може виконати таку задачу, якщо описати її в промпті, але:
- вона не знає переліку міток компанії, тому його доводиться передавати в кожному запиті;
- навіть із переліком вона часто плутає близькі мітки, бо не знає, як саме компанія їх розрізняє
  (наприклад, `get_refund` і `track_refund`).

Донавчання на прикладах із правильними мітками закріплює формат відповіді й значення кожної мітки у вагах моделі,
тому після нього достатньо короткого промпту.

План:
1. Підготувати дані у форматі `messages` і розбити їх на тренувальний, валідаційний і тестовий набори.
2. Підготувати функції для генерації й оцінювання.
3. Оцінити, чому повне донавчання великої моделі не вміщується в GPU T4.
4. Налаштувати **LoRA** і параметри тренування.
5. Донавчити методом LoRA моделі **Qwen3-0.6B, Qwen3-1.7B і Qwen3-4B** і порівняти, чи може менша модель після донавчання наздогнати більшу.
6. Виконати **повне донавчання** найменшої моделі й порівняти його з LoRA.
7. Зберегти адаптер, завантажити його знову й влити у ваги моделі.
8. Донавчити Qwen3-4B методом **QLoRA** і порівняти витрати памʼяті.

Приклад за мотивами розділу 11.4 [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/4)
і вправ курсу [smol-course](https://huggingface.co/learn/smol-course).
Для запуску в Colab оберіть середовище з GPU.

In [1]:
# trl — бібліотека для донавчання мовних моделей (SFT, DPO тощо),
# peft — реалізація LoRA та інших методів PEFT, bitsandbytes — 4-бітна квантизація для QLoRA
!pip install -q "transformers>=4.55.0" "trl>=1.17" "peft" "bitsandbytes" "datasets" "accelerate"
# Colab має застарілу бібліотеку torchao, з якою peft не імпортується; для цього демо вона не потрібна
!pip uninstall -q -y torchao

In [2]:
import gc
import json
import os
import time

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Для цього ноутбука потрібна GPU"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Памʼять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: Tesla T4
Памʼять GPU: 15.6 GB


## 1. Дані

[Bitext Customer Support](https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset) —
датасет із майже 27 тис. повідомлень клієнтів англійською мовою. Кожне повідомлення має дві мітки:
- `category` — відділ, якого стосується звернення (11 категорій, наприклад `ORDER`, `REFUND`, `ACCOUNT`);
- `intent` — конкретний намір клієнта (27 намірів, наприклад `cancel_order`, `track_refund`).

Конкретні дані в повідомленнях замінено заповнювачами на кшталт `{{Order Number}}`.

In [3]:
raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train")
print(raw_dataset)

for ex in raw_dataset.shuffle(seed=0).select(range(5)):
    print(f"{ex['category']:<10} {ex['intent']:<25} {ex['instruction']}")

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Dataset({
    features: ['flags', 'instruction', 'category', 'intent', 'response'],
    num_rows: 26872
})
SHIPPING   set_up_shipping_address   there are errors setting up a delivery address
PAYMENT    check_payment_methods     i want assistance seeing ur accepted payment opotions
SUBSCRIPTION newsletter_subscription   I have got to cancel the newsletter subscription
ACCOUNT    switch_account            can ya help me to change to the gold account
REFUND     track_refund              I am waiting for a compensation of {{Currency Symbol}}{{Refund Amount}}


In [4]:
categories = sorted(set(raw_dataset["category"]))
intents = sorted(set(raw_dataset["intent"]))
print(f"Категорій: {len(categories)}, намірів: {len(intents)}")
print(categories)

Категорій: 11, намірів: 27
['ACCOUNT', 'CANCEL', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICE', 'ORDER', 'PAYMENT', 'REFUND', 'SHIPPING', 'SUBSCRIPTION']


### Дедуплікація

Частина звернень у датасеті повторюється дослівно. Якщо такий дублікат потрапить і в тренувальний, і у валідаційний набір,
validation loss буде заниженим: модель перевірятимуть на прикладах, які вона вже бачила.
Тому перед розбиттям залишаємо по одному прикладу для кожного тексту звернення (без урахування регістру й пробілів на краях).

In [5]:
seen, keep = set(), []
for i, text in enumerate(raw_dataset["instruction"]):
    key = text.strip().lower()
    if key not in seen:
        seen.add(key)
        keep.append(i)

dedup_dataset = raw_dataset.select(keep)
print(f"До дедуплікації: {len(raw_dataset)}, після: {len(dedup_dataset)}")

До дедуплікації: 26872, після: 24371


### Формат `messages`

Кожне звернення перетворюємо на діалог із трьох повідомлень:
- `system` — коротка інструкція;
- `user` — повідомлення клієнта;
- `assistant` — правильна відповідь: JSON із мітками.

Колонка `chat_template_kwargs` вимикає режим міркувань Qwen3: модель має одразу відповідати JSON.

In [6]:
SHORT_PROMPT = "Classify the customer message. Reply with JSON only."


def to_messages(example):
    answer = json.dumps({"category": example["category"], "intent": example["intent"]})
    return {
        "messages": [
            {"role": "system", "content": SHORT_PROMPT},
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": answer},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

### Тренувальний, валідаційний і тестовий набори

- **Тренувальний** (2800 прикладів) — на ньому модель вчиться.
- **Валідаційний** (200) — на ньому під час тренування обчислюється validation loss, щоб помітити перенавчання.
- **Тестовий** (1000) — на ньому лише один раз, після тренування, рахуємо підсумкові метрики.
  Під час тренування й налаштування ці приклади не використовуються, тому оцінка на них найчесніша.

Після дедуплікації та перемішування тестовий набір беремо з прикладів, які не потрапили до перших 3000.

Тестовий набір більший за валідаційний, бо від його розміру залежить точність вимірювання метрик:
на 1000 прикладах одна помилка змінює точність на 0,1%, тоді як на 200 — на 0,5%.
Донавчені моделі помиляються рідко, і на малому тестовому наборі різниця між ними може бути непомітною.

In [7]:
shuffled = dedup_dataset.shuffle(seed=42)

dataset = shuffled.select(range(3000)).map(to_messages, remove_columns=shuffled.column_names)
dataset = dataset.train_test_split(test_size=200, seed=42)
train_dataset, val_dataset = dataset["train"], dataset["test"]
test_dataset = shuffled.select(range(3000, 4000)).map(to_messages, remove_columns=shuffled.column_names)

print(f"Тренувальний: {len(train_dataset)}, валідаційний: {len(val_dataset)}, тестовий: {len(test_dataset)}")
train_dataset[0]["messages"]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Тренувальний: 2800, валідаційний: 200, тестовий: 1000


[{'content': 'Classify the customer message. Reply with JSON only.',
  'role': 'system'},
 {'content': 'I have got to locate hte bills from {{Person Name}}',
  'role': 'user'},
 {'content': '{"category": "INVOICE", "intent": "check_invoice"}',
  'role': 'assistant'}]

## 2. Функції для генерації й оцінювання

Порівнюємо три моделі однієї сімʼї з однаковою архітектурою й даними попереднього тренування — **Qwen3** на 0,6, 1,7 і 4 млрд параметрів.
Усі вони мають той самий токенізатор і шаблон розмови, тому токенізатор завантажуємо один раз.

In [9]:
MODELS = ["Qwen/Qwen3-0.6B", "Qwen/Qwen3-1.7B", "Qwen/Qwen3-4B"]

tokenizer = AutoTokenizer.from_pretrained(MODELS[0])

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

### Детальний промпт

Без донавчання модель не знає переліку міток, тому для оцінки «до» передаємо його в промпті разом з описом формату.

In [10]:
LONG_PROMPT = f"""Classify the customer message for the support ticket routing system.
Reply with a single JSON object and nothing else, in the format:
{{"category": "<CATEGORY>", "intent": "<intent>"}}

Allowed categories: {", ".join(categories)}
Allowed intents: {", ".join(intents)}"""

for name, prompt in [("Короткий промпт", SHORT_PROMPT), ("Детальний промпт", LONG_PROMPT)]:
    print(f"{name}: {len(tokenizer(prompt)['input_ids'])} токенів")

Короткий промпт: 11 токенів
Детальний промпт: 166 токенів


### Метрики

Відповідь моделі вважаємо правильною, якщо програма може розібрати її як JSON і мітки збігаються з еталонними.
До донавчання моделі Qwen3 часто загортають JSON у блок розмітки Markdown (` ```json … ``` `), тому таку обгортку перед розбором знімаємо,
але окремо рахуємо, як часто відповідь є чистим JSON без зайвого тексту.

Рахуємо чотири метрики на всьому тестовому наборі:
- частку відповідей, які є чистим JSON;
- частку відповідей, які після зняття обгортки є валідним JSON з полями `category` і `intent`;
- точність категорії;
- точність наміру.

Щоб оцінка 1000 прикладів не тривала надто довго, відповіді генеруються батчами по 32 звернення.

In [11]:
def classify_batch(model, messages_batch, system_prompt, max_new_tokens=60):
    """Повертає відповіді моделі на список звернень клієнтів, згенеровані одним батчем."""
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for message in messages_batch
    ]
    # Для генерації батчем короткі промпти доповнюються зліва, щоб відповіді всіх промптів починалися в одній позиції
    tokenizer.padding_side = "left"
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    tokenizer.padding_side = "right"
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id
        )
    new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
    return [answer.strip() for answer in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]


def classify(model, message, system_prompt):
    """Повертає відповідь моделі на одне звернення клієнта."""
    return classify_batch(model, [message], system_prompt)[0]


def is_clean_json(answer):
    """Чи є відповідь JSON без жодного зайвого тексту."""
    try:
        json.loads(answer)
        return True
    except json.JSONDecodeError:
        return False


def parse(answer):
    """Знімає обгортку ```json … ```, розбирає JSON; повертає None, якщо формат неправильний."""
    text = answer.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        return None
    if isinstance(result, dict) and {"category", "intent"} <= result.keys():
        return result
    return None


def evaluate(model, system_prompt, batch_size=32):
    """Рахує метрики на всьому тестовому наборі."""
    model.eval()
    messages = [ex["messages"][1]["content"] for ex in test_dataset]
    targets = [json.loads(ex["messages"][2]["content"]) for ex in test_dataset]
    answers = []
    for start in range(0, len(messages), batch_size):
        answers += classify_batch(model, messages[start:start + batch_size], system_prompt)

    clean = valid = category_ok = intent_ok = 0
    for answer, target in zip(answers, targets):
        clean += is_clean_json(answer)
        parsed = parse(answer)
        if parsed is not None:
            valid += 1
            category_ok += parsed["category"] == target["category"]
            intent_ok += parsed["intent"] == target["intent"]
    n = len(answers)
    metrics = {
        "Чистий JSON, %": 100 * clean / n,
        "Валідний JSON, %": 100 * valid / n,
        "Точність category, %": 100 * category_ok / n,
        "Точність intent, %": 100 * intent_ok / n,
    }
    return metrics, list(zip(messages, answers))


def show(answers, k=5):
    """Друкує перші k пар «звернення — відповідь моделі»."""
    for message, answer in answers[:k]:
        print(f"📩 {message}\n🤖 {answer}\n" + "-" * 60)


def free_memory():
    """Звільняє памʼять GPU після видалення моделей."""
    gc.collect()
    torch.cuda.empty_cache()

## 3. Чому не повне донавчання

Під час повного донавчання з оптимізатором Adam у змішаній точності на кожен параметр потрібно приблизно 16 байт:

| Що зберігається | Байтів на параметр |
|---|---|
| ваги у fp16 | 2 |
| градієнти у fp16 | 2 |
| копія ваг у fp32: у fp16 малі оновлення ваг губляться через округлення | 4 |
| два стани Adam у fp32: усереднені градієнт і його квадрат, за якими Adam підбирає крок для кожного параметра | 8 |
| **разом** | **16** |

До цього додаються активації, які залежать від довжини послідовностей і розміру батча.

Для Qwen3-4B це близько 64 GB без активацій, тоді як T4 має 15 GB: повне донавчання такої моделі потребує GPU рівня A100 на 80 GB.
Для Qwen3-0.6B потрібно лише близько 10 GB, тому в розділі 6 виконаємо повне донавчання саме цієї моделі.

## 4. Налаштування LoRA і тренування

LoRA заморожує всі ваги моделі й додає до вибраних шарів дві невеликі матриці $A$ і $B$.
Тренуються лише вони, тому градієнти й стани оптимізатора потрібні тільки для них.

Параметри `LoraConfig`:
- `r` — ранг матриць адаптера: що більший, то більше параметрів і «свободи» має адаптер;
- `lora_alpha` — масштаб внеску адаптера (зазвичай `2 × r`);
- `lora_dropout` — dropout в адаптері для регуляризації;
- `target_modules="all-linear"` — адаптери додаються до всіх лінійних шарів (attention і MLP).

In [12]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

Параметри тренування (`SFTConfig`):
- `assistant_only_loss=True` — loss рахується лише на відповідях асистента, а не на репліках користувача й системному промпті:
  для решти токенів мітки замінюються значенням −100, і функція втрат їх пропускає;
- `max_steps=100` — кількість кроків тренування; за один крок модель бачить 8 прикладів, тобто за все тренування — 800 із 2800,
  менше третини епохи. Для вузької задачі цього достатньо: validation loss виходить на плато раніше;
- `learning_rate=2e-4` — для LoRA learning rate приблизно в 10 разів більший, ніж для повного донавчання;
- `eval_strategy="steps"` і `eval_steps=10` — кожні 10 кроків модель оцінюється на валідаційному наборі:
  якщо validation loss почне зростати, модель перенавчається;
- `fp16=True` — змішана точність (T4 не підтримує bf16);
- `gradient_checkpointing=True` — активації не зберігаються, а перераховуються під час зворотного проходу: повільніше, але потрібно менше памʼяті.

In [13]:
def make_config(output_dir, learning_rate=2e-4):
    return SFTConfig(
        output_dir=output_dir,
        max_length=256,                  # діалог із цього датасету після шаблону займає близько 100 токенів
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,   # ефективний розмір батча: 4 × 2 = 8 прикладів
        per_device_eval_batch_size=8,
        learning_rate=learning_rate,
        lr_scheduler_type="cosine",
        warmup_steps=5,
        max_steps=100,
        eval_strategy="steps",
        eval_steps=10,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        assistant_only_loss=True,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
    )


def train(model, output_dir, peft_config=peft_config, learning_rate=2e-4):
    """Тренує модель (з LoRA, якщо задано peft_config) і повертає тренер, тривалість, пікову памʼять GPU та validation loss."""
    torch.cuda.reset_peak_memory_stats()
    trainer = SFTTrainer(
        model=model,
        args=make_config(output_dir, learning_rate),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # Для QLoRA бібліотека TRL переводить ваги адаптера в bf16, який T4 не підтримує.
    # Повертаємо їх у fp32: змішана точність fp16 потребує, щоб параметри, що тренуються, були у fp32
    if getattr(model, "is_loaded_in_4bit", False):
        for param in trainer.model.parameters():
            if param.requires_grad:
                param.data = param.data.float()
    if peft_config is not None:
        trainer.model.print_trainable_parameters()
    start = time.time()
    trainer.train()
    eval_losses = [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h]
    stats = {
        "Тривалість, хв": (time.time() - start) / 60,
        "Пікова памʼять GPU, GB": torch.cuda.max_memory_allocated() / 1e9,
        "Validation loss": eval_losses[-1],
    }
    return trainer, stats

## 5. LoRA на моделях різного розміру

Для кожної моделі виконуємо однакові кроки:
1. завантажуємо модель у fp16 (ваги Qwen3 збережено в bfloat16, але T4 не підтримує його апаратно);
2. оцінюємо її **до донавчання** з детальним промптом, у якому перелічено всі мітки;
3. донавчаємо методом LoRA з однаковими параметрами;
4. оцінюємо **після донавчання** з коротким промптом без переліку міток;
5. зберігаємо адаптер і вивантажуємо модель з GPU, щоб звільнити памʼять для наступної.

In [14]:
results = {}


def run_lora(model_name):
    """Повний цикл для однієї моделі: оцінка до, LoRA, оцінка після, збереження адаптера."""
    short_name = model_name.split("/")[-1]
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16, device_map="auto")
    print(f"{short_name}: {model.num_parameters() / 1e9:.2f} млрд параметрів, "
          f"памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

    before_metrics, before_answers = evaluate(model, LONG_PROMPT)
    print("\nДо донавчання:")
    show(before_answers, k=3)

    trainer, stats = train(model, f"{short_name}-lora")
    after_metrics, after_answers = evaluate(trainer.model, SHORT_PROMPT)
    print("\nПісля донавчання:")
    show(after_answers, k=3)

    adapter_dir = f"{short_name}-lora-adapter"
    trainer.save_model(adapter_dir)
    stats["Адаптер, MB"] = os.path.getsize(os.path.join(adapter_dir, "adapter_model.safetensors")) / 1e6

    results[short_name] = {"до": before_metrics, "LoRA": after_metrics | stats}
    del trainer, model
    free_memory()
    return pd.DataFrame(results[short_name]).round(2)

In [15]:
run_lora("Qwen/Qwen3-0.6B")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3-0.6B: 0.60 млрд параметрів, памʼять GPU після завантаження: 1.2 GB

До донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "CONTACT", "intent": "contact_customer_service"}
------------------------------------------------------------


Tokenizing train dataset:   0%|          | 0/2800 [00:00<?, ? examples/s]

... (7 outputs left out)

trainable params: 10,092,544 || all params: 606,142,464 || trainable%: 1.6650


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.362360,0.477780,0.437490,0.894284,4323.000000
20,0.321424,0.142513,0.187870,0.956574,8672.000000
30,0.080906,0.061217,0.072918,0.985931,13032.000000
40,0.058993,0.026447,0.043448,0.991562,17377.000000
50,0.024378,0.011455,0.029028,0.996862,21714.000000
60,0.006417,0.007596,0.018816,0.998415,26070.000000
70,0.011626,0.006467,0.016692,0.998124,30452.000000
80,0.007643,0.005398,0.013868,0.998745,34813.000000
90,0.005070,0.005611,0.013740,0.998755,39116.000000
100,0.015931,0.005526,0.013644,0.998755,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",37.9,99.70
"Точність intent, %",25.6,97.90
"Тривалість, хв",NaN,2.55
"Пікова памʼять GPU, GB",NaN,1.42
Validation loss,NaN,0.01
"Адаптер, MB",NaN,40.42


In [16]:
run_lora("Qwen/Qwen3-1.7B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.993910,0.438160,0.368747,0.905950,4323.000000
20,0.316205,0.134141,0.193240,0.955362,8672.000000
30,0.080266,0.045413,0.050295,0.988424,13032.000000
40,0.049677,0.024782,0.028430,0.990952,17377.000000
50,0.017400,0.017461,0.025658,0.994078,21714.000000
60,0.012535,0.005518,0.014731,0.998740,26070.000000
70,0.013604,0.005301,0.011386,0.997494,30452.000000
80,0.010694,0.005120,0.010532,0.997494,34813.000000
90,0.013733,0.004267,0.010443,0.997812,39116.000000
100,0.009780,0.004063,0.010375,0.997812,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",62.7,99.30
"Точність intent, %",57.4,96.70
"Тривалість, хв",NaN,2.55
"Пікова памʼять GPU, GB",NaN,3.83
Validation loss,NaN,0.00
"Адаптер, MB",NaN,69.78


In [17]:
run_lora("Qwen/Qwen3-4B")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (2 outputs left out)

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.768315,0.299888,0.306240,0.923112,4323.000000
20,0.202471,0.100536,0.100264,0.962126,8672.000000
30,0.065335,0.029527,0.035166,0.989987,13032.000000
40,0.035080,0.018715,0.026599,0.995031,17377.000000
50,0.014228,0.007152,0.015737,0.998422,21714.000000
60,0.008851,0.003758,0.009342,0.998437,26070.000000
70,0.007472,0.005168,0.008858,0.998117,30452.000000
80,0.012325,0.004774,0.008141,0.998110,34813.000000
90,0.015573,0.003419,0.008094,0.998742,39116.000000
100,0.011943,0.003314,0.008018,0.998742,43449.000000



Після донавчання:
📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,до,LoRA
"Чистий JSON, %",100.0,100.00
"Валідний JSON, %",100.0,100.00
"Точність category, %",88.1,100.00
"Точність intent, %",72.5,97.90
"Тривалість, хв",NaN,3.65
"Пікова памʼять GPU, GB",NaN,8.75
Validation loss,NaN,0.00
"Адаптер, MB",NaN,132.19


### Порівняння

Чи може менша модель після донавчання досягти якості більшої? Порівняйте точність, тривалість тренування, памʼять і розмір адаптера.

In [18]:
pd.DataFrame({
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00
"Точність intent, %",25.6,97.90,57.4,96.70,72.5,97.90
"Тривалість, хв",NaN,2.55,NaN,2.55,NaN,3.65
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19


## 6. Повне донавчання Qwen3-0.6B

Модель на 0,6 млрд параметрів потребує для повного донавчання близько 10 GB, тому на T4 її можна донавчити повністю й порівняти з LoRA.

Відмінності від LoRA:
- `peft_config=None` — тренуються всі ваги моделі;
- модель завантажуємо у **fp32**: змішана точність fp16 тримає головну копію ваг у fp32, а в fp16 лише обчислює прохід;
- `learning_rate=2e-5` — для повного донавчання learning rate приблизно в 10 разів менший, ніж для LoRA.

In [19]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B", dtype=torch.float32, device_map="auto")
trainer, full_stats = train(model, "Qwen3-0.6B-full", peft_config=None, learning_rate=2e-5)
full_metrics, full_answers = evaluate(trainer.model, SHORT_PROMPT)
show(full_answers, k=3)

del trainer, model
free_memory()

pd.DataFrame({"Qwen3-0.6B LoRA": results["Qwen3-0.6B"]["LoRA"], "Qwen3-0.6B повне": full_metrics | full_stats}).round(2)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.255744,0.288718,0.251618,0.934379,4323.000000
20,0.176103,0.060587,0.091798,0.985634,8672.000000
30,0.049239,0.038266,0.028003,0.990942,13032.000000
40,0.046789,0.025577,0.022165,0.994027,17377.000000
50,0.026981,0.007719,0.010975,0.998120,21714.000000
60,0.002138,0.004414,0.004073,0.998442,26070.000000
70,0.005830,0.004066,0.003160,0.999375,30452.000000
80,0.012958,0.003836,0.004107,0.998437,34813.000000
90,0.009277,0.003028,0.003516,0.999370,39116.000000
100,0.014481,0.002767,0.003431,0.999370,43449.000000


📩 help me to inform of troubles with online payments
🤖 {"category": "PAYMENT", "intent": "payment_issue"}
------------------------------------------------------------
📩 i need assistance sending my feedback for a product
🤖 {"category": "FEEDBACK", "intent": "review"}
------------------------------------------------------------
📩 help me report problems with a signup
🤖 {"category": "ACCOUNT", "intent": "registration_problems"}
------------------------------------------------------------


,Qwen3-0.6B LoRA,Qwen3-0.6B повне
"Чистий JSON, %",100.00,100.00
"Валідний JSON, %",100.00,100.00
"Точність category, %",99.70,99.50
"Точність intent, %",97.90,98.30
"Тривалість, хв",2.55,1.68
"Пікова памʼять GPU, GB",1.42,10.80
Validation loss,0.01,0.00
"Адаптер, MB",40.42,NaN


## 7. Збереження і завантаження адаптера

`save_model` зберігає **лише адаптер** (`adapter_model.safetensors`) разом із його налаштуваннями й токенізатором, а не всю модель.

In [20]:
adapter_dir = "Qwen3-4B-lora-adapter"
for name in sorted(os.listdir(adapter_dir)):
    size = os.path.getsize(os.path.join(adapter_dir, name))
    print(f"{name:<35} {size / 1e6:>8.1f} MB")
print("\nДля порівняння, ваги Qwen3-4B у fp16: ~8 GB")

README.md                                0.0 MB
adapter_config.json                      0.0 MB
adapter_model.safetensors              132.2 MB
chat_template.jinja                      0.0 MB
tokenizer.json                          11.4 MB
tokenizer_config.json                    0.0 MB
training_args.bin                        0.0 MB

Для порівняння, ваги Qwen3-4B у fp16: ~8 GB


Щоб використати адаптер, завантажуємо модель і накладаємо на неї адаптер з диска.
Метод `merge_and_unload` вливає адаптер у ваги моделі: $W' = W + \frac{\alpha}{r} BA$.
Після цього модель працює як звичайна, без додаткових обчислень для адаптера.

In [21]:
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_dir)
model = model.merge_and_unload()
print(type(model).__name__)

# Злиту модель можна зберегти як звичайну: model.save_pretrained("qwen3-4b-merged")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM


Перевіримо злиту модель на власних повідомленнях:

In [22]:
my_messages = [
    "Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?",
    "I was charged twice for the same order, please help",
    "How can I talk to a real person?",
    "I guess, you have shipped to a wrong address",
]
for message in my_messages:
    print(f"📩 {message}\n🤖 {classify(model, message, SHORT_PROMPT)}\n")

📩 Hi, I ordered a laptop last week and it still hasn't arrived. Where is it?
🤖 {"category": "DELIVERY", "intent": "delivery_period"}

📩 I was charged twice for the same order, please help
🤖 {"category": "ORDER", "intent": "cancel_order"}

📩 How can I talk to a real person?
🤖 {"category": "CONTACT", "intent": "contact_human_agent"}

📩 I guess, you have shipped to a wrong address
🤖 {"category": "SHIPPING", "intent": "shipping_address"}



In [23]:
del model
free_memory()

## 8. QLoRA

QLoRA завантажує заморожені ваги Qwen3-4B в **4 бітах** (формат NF4), а адаптери LoRA тренує у звичайній точності.
Код тренування той самий, змінюється лише завантаження моделі.

### Формат NF4

У 4 бітах вага може набувати лише одного з 16 значень. Ваги ділять на блоки по 64, і для кожного блоку
зберігають масштаб — найбільше за модулем значення в блоці. Формати відрізняються тим, як розташовані ці 16 значень:
- **INT4** — 16 рівновіддалених значень. Ваги нейромережі розподілені приблизно нормально: більшість близька до нуля,
  а великих за модулем мало. Тому крайні значення використовуються рідко, а біля нуля їх бракує;
- **NF4 (NormalFloat4)** — 16 значень розташовані за квантилями нормального розподілу: густо біля нуля й рідко на краях.
  Кожне значення використовується приблизно однаково часто, тому помилка квантизації менша, ніж у INT4.

NF4 — формат лише для зберігання: перед кожним множенням ваги деквантизуються у fp16 (`bnb_4bit_compute_dtype`).
Детальніше — у блозі Hugging Face [Making LLMs even more accessible with bitsandbytes, 4-bit quantization and QLoRA](https://huggingface.co/blog/4bit-transformers-bitsandbytes)
і в статті [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314).

In [24]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-бітний формат, розрахований на нормально розподілені ваги
    bnb_4bit_compute_dtype=torch.float16,   # обчислення виконуються у fp16
    bnb_4bit_use_double_quant=True,         # квантизуються також масштаби квантизації: ще трохи менше памʼяті
)

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", quantization_config=bnb_config, device_map="auto")
print(f"Памʼять GPU після завантаження: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Памʼять GPU після завантаження: 2.7 GB


In [25]:
trainer, qlora_stats = train(model, "Qwen3-4B-qlora")
qlora_metrics, _ = evaluate(trainer.model, SHORT_PROMPT)
qlora_stats

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.482087,0.316073,0.297632,0.923437,4323.000000
20,0.196355,0.128010,0.097375,0.957768,8672.000000
30,0.071827,0.025940,0.042633,0.993133,13032.000000
40,0.029084,0.014809,0.022656,0.996249,17377.000000
50,0.016078,0.012506,0.016101,0.996207,21714.000000
60,0.008284,0.005340,0.008323,0.998728,26070.000000
70,0.006567,0.006163,0.007853,0.998422,30452.000000
80,0.003313,0.006240,0.007128,0.997485,34813.000000
90,0.013288,0.004978,0.007196,0.998422,39116.000000
100,0.011276,0.004774,0.007221,0.998422,43449.000000


{'Тривалість, хв': 4.947020904223124,
 'Пікова памʼять GPU, GB': 3.665665536,
 'Validation loss': 0.00477382680401206}

## 9. Підсумкове порівняння

In [26]:
summary = {
    f"{name} {stage}": metrics
    for name, stages in results.items()
    for stage, metrics in stages.items()
}
summary["Qwen3-0.6B повне"] = full_metrics | full_stats
summary["Qwen3-4B QLoRA"] = qlora_metrics | qlora_stats
pd.DataFrame(summary).round(2)

,Qwen3-0.6B до,Qwen3-0.6B LoRA,Qwen3-1.7B до,Qwen3-1.7B LoRA,Qwen3-4B до,Qwen3-4B LoRA,Qwen3-0.6B повне,Qwen3-4B QLoRA
"Чистий JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Валідний JSON, %",100.0,100.00,100.0,100.00,100.0,100.00,100.00,100.00
"Точність category, %",37.9,99.70,62.7,99.30,88.1,100.00,99.50,99.90
"Точність intent, %",25.6,97.90,57.4,96.70,72.5,97.90,98.30,97.80
"Тривалість, хв",NaN,2.55,NaN,2.55,NaN,3.65,1.68,4.95
"Пікова памʼять GPU, GB",NaN,1.42,NaN,3.83,NaN,8.75,10.80,3.67
Validation loss,NaN,0.01,NaN,0.00,NaN,0.00,0.00,0.00
"Адаптер, MB",NaN,40.42,NaN,69.78,NaN,132.19,NaN,NaN


**Що варто помітити:**
- LoRA тренує близько 1–2% параметрів моделі, а адаптер займає десятки або сотні мегабайт замість гігабайтів для всієї моделі.
- Порівняйте моделі різного розміру: наскільки менша модель після донавчання поступається більшій і скільки вона заощаджує часу й памʼяті.
  Для вузької задачі з тисячами прикладів донавчена мала модель може бути вигіднішою за велику.
- Порівняйте повне донавчання Qwen3-0.6B з LoRA на тій самій моделі за якістю, часом і памʼяттю.
- До донавчання моделі отримують детальний промпт із переліком міток, після — лише короткий:
  формат і значення міток тепер зберігаються у вагах, а не в промпті. Коротший промпт означає швидшу й дешевшу обробку кожного запиту.
- QLoRA потребує помітно менше памʼяті, ніж LoRA на тій самій моделі, але тренується повільніше, бо ваги щоразу деквантизуються під час обчислень.

## 0. Опрацюйте демо-ноутбук лекції 6

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука в Google Colab.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (відповіді на питання потрібно написати у демо-ноутбуці)

22. Як довго виконувалися клітинки ноутбуку 03_sft_lora? Чи були проблеми з запуском, які?
23. Прокоментуйте вивід "Памʼять GPU після завантаження". Чи очікуваним для Вас є результат?
24. Що робить рядок with torch.no_grad() у функції classify()?
25. Для чого у classify() задано do_sample=False?
26. В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?
27. Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?
28. Для чого в датасеті задано колонку chat_template_kwargs з enable_thinking=False? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі /think?
29. Що означає параметр assistant_only_loss=True? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?
30. Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?

### 0.2. Мої відповіді:

22. Як довго виконувалися клітинки ноутбуку 03_sft_lora? Чи були проблеми з запуском, які?

- **Встановлення бібліотек** - 9 секунд.
- **Імпорт** - 51 секунда.
- **Оцінка «до» + LoRA-тренування + оцінка «після»** для трьох моделей:
  - Qwen3-0.6B - **4 хвилини**.
  - Qwen3-1.7B - **8 хвилин**.
  - Qwen3-4B - **16 хвилин**.
- **Повне донавчання Qwen3-0.6B** - 2 хвилини.
- **QLoRA Qwen3-4B** - **7 хвилин**.
- **Завантаження й злиття адаптера** - 34 секунди.
- Проблем не виявлено

23. Прокоментуйте вивід "Памʼять GPU після завантаження". Чи очікуваним для Вас є результат?

У виводі клітинки 24 для Qwen3-4B у 4-бітному форматі (NF4) показано: **памʼять GPU після завантаження - ~2.5 GB**.

**Це цілком очікувано.**

- Qwen3-4B у fp16 займає **~8 GB** (4 млрд × 2 байти).
- У 4-бітному NF4 - **~2 GB** (4 млрд × 0.5 байти), плюс overhead на масштаби й буфери.
- Фактичні **~2.5 GB** - це ваги + трохи пам'яті на квантизаційні масштаби (double quant) і буфери.

**Висновок:** QLoRA дає змогу завантажити 4B-модель у 4 рази менше пам'яті (8 GB → 2.5 GB), що критично для T4 з 15 GB. Без QLoRA повне донавчання 4B потребувало б ~64 GB.

24. Що робить рядок with torch.no_grad() у функції classify()?

`torch.no_grad()` **вимикає обчислення градієнтів** під час прямого проходу.

Це означає:
- PyTorch **не будує граф обчислень** (не зберігає проміжні тензори для зворотного проходу).
- **Економить пам'ять** (не треба зберігати активації для backprop).
- **Прискорює** обчислення (менше накладних витрат).

Під час **інференсу** (генерації відповіді) ми **не тренуємо** модель, тому градієнти не потрібні. Без `no_grad()` PyTorch будував би граф для кожного токена, що споживало б багато пам'яті й було б повільніше.

25. Для чого у classify() задано do_sample=False?

`do_sample=False` вмикає **жадібну генерацію (greedy decoding)**: модель завжди обирає **найімовірніший** наступний токен, без випадковості.

- **Детермінованість:** на одне й те саме звернення модель завжди дасть **однакову** відповідь. Це критично для **оцінки** (порівнюємо моделі на одних і тих самих прикладах).
- **Стабільність JSON:** для задачі класифікації нам потрібен **точний формат** (`{"category": ..., "intent": ...}`). Випадковість могла б зламати JSON (наприклад, додати зайвий текст).
- **Швидкість:** greedy decoding працює швидше, ніж sampling (не треба рахувати ймовірності для випадкового вибору).

**`do_sample=True`** для творчих задач, де потрібна різноманітність, але **не для класифікації**.

26. В демо 1 було натреновано класифікатор з базової моделі за допомогою заміни голови. В демо 3 в ході інференсу моделі також відбувається класифікація звернень. Чому не можна застосувати такий же підхід?

Тому що це **різні типи задач**.

- **Демо 1 (BERT + нова голова):** класифікація **пари речень** (2 класи: парафраз / не парафраз). Голова - це **один лінійний шар**. Це **проста бінарна класифікація**.
- **Демо 3 (SFT з LoRA):** класифікація **звернень клієнтів** у **JSON-формат** із **двома полями** (category з 11 варіантів + intent з 27 варіантів). Це **не бінарна класифікація**, а **структурована генерація**.

**Чому не можна замінити голову:**
1. **Кількість класів:** у демо 3 - **11 × 27 = 297 комбінацій** (category × intent). Голова на 297 виходів була б **величезною** й важкою для тренування.
2. **Формат:** нам потрібен **текстовий JSON**, а не просто номер класу. Голова не вміє генерувати текст - вона видає логіти.
3. **Генерація:** демо 3 - це **generative task** (модель пише текст), а не **classification task** (модель обирає клас). Для генерації потрібна **повна LLM**, а не голова.


27. Порівняйте кількість токенів короткого й детального промпту (розділ 2). Як ця різниця вплине на вартість обробки 10 000 звернень на день?

У виводі клітинки 10:
- **Короткий промпт** (`SHORT_PROMPT`): **11 токенів**.
- **Детальний промпт** (`LONG_PROMPT`): **166 токенів**.

**Різниця:** детальний промпт у **15 разів довший** (166 / 11 ≈ 15).

**Вплив на вартість (10 000 звернень на день):**
- Якщо платити за токени (наприклад, у хмарному API):
  - Короткий: 10 000 × 11 = **110 000 токенів/день** на промпт.
  - Детальний: 10 000 × 166 = **1 660 000 токенів/день** на промпт.
  - **Різниця - 1 550 000 токенів/день** (у 15 разів більше).
- Якщо локально (на своєму залізі):
  - Детальний промпт **довше обробляється** (prefill), що збільшує **TTFT** (time to first token) і споживає більше пам'яті на KV-кеш.

Але після SFT модель **не потребує** детального промпту. Вона **знає** мітки з ваг. Тому короткий промпт (11 токенів) **у 15 разів дешевший** і швидший. Це і є **головна економічна вигода** донавчання: ми «вбудовуємо» знання в модель, а не передаємо їх у кожному запиті.

28. Для чого в датасеті задано колонку chat_template_kwargs з enable_thinking=False? Що, на Вашу думку, сталося б із моделлю, якби її донавчали на цих прикладах у режимі /think?

`chat_template_kwargs={"enable_thinking": False}` **вимикає режим міркувань** Qwen3 під час застосування шаблону. Це означає, що модель **не генерує блок `<think>…</think>`**, а одразу видає JSON.

**Навіщо це:**
- Задача - **класифікація** (швидко, чітко, JSON). Міркування тут **не потрібні**, бо вони лише додають токени й час.
- Без `enable_thinking=False` модель генерувала б спочатку міркування, а потім JSON - це **у 5–10 разів більше токенів**.

**Що сталося б у режимі `/think`:**
1. **Довші відповіді:** модель генерувала б `<think>...</think>` перед JSON.
2. **Повільніше:** більше токенів = довше чекати.
3. **Дорожче:** якщо платити за токени - у 5–10 разів дорожче.
4. **Ризик зламати JSON:** міркування могло б «зіпсувати» формат (наприклад, модель вирішила б додати коментар після JSON).
5. **Модель навчилася б марно міркувати:** на простих задачах класифікації це неефективно.

29. Що означає параметр assistant_only_loss=True? На яких токенах модель вчилася б без нього і чим це погано для нашої задачі?

`assistant_only_loss=True` означає, що **loss рахується лише на токенах відповіді асистента**. Для решти токенів (system + user) мітки замінюються на `-100`, і функція втрат їх **ігнорує**.

**Без `assistant_only_loss=True` (тобто `False`):**

Модель вчилася б на **всіх** токенах:
- **System-промпт** («Classify the customer message...»).
- **User-повідомлення** (звернення клієнта).
- **Assistant-відповідь** (JSON).

**Чим це погано для нашої задачі:**
- **Модель вчилася б генерувати system-промпт**, але це не її робота (це інструкція для неї).
- **Модель вчилася б генерувати user-повідомлення**, але їх пише клієнт, а не модель.
- **Loss «розмивався» б:** більша частина токенів це system і user, тому модель більше уваги приділяла б їм, а не відповіді.
- **Перенавчання на промптах:** модель могла б почати «повторювати» промпт у відповіді.

30. Порівняйте LoRA і QLoRA за пам'яттю, часом тренування і якістю (розділ 9). У яких випадках Ви обрали б QLoRA?

 За результатами розділу 9 (таблиця з виводу):

| Показник | Qwen3-4B LoRA | Qwen3-4B QLoRA |
|---|---|---|
| Точність category | **100%** | **99.9%** |
| Точність intent | **97.9%** | 97.8% |
| Пікова пам'ять GPU | **8.75 GB** | **3.67 GB** |
| Тривалість тренування | **3.65 хв** | 4.95 хв |
| Validation loss | 0.00 | 0.00 |

**Детальніше:**
- **Пам'ять:** QLoRA потребує **у 2.4 раза менше** пам'яті (3.67 GB проти 8.75 GB), бо ваги зберігаються в 4 бітах (NF4). Це критично для T4 з 15 GB - LoRA на 4B (8.75 GB) залишає мало місця для активацій.
- **Час:** QLoRA **повільніше** (4.95 хв проти 3.65 хв), бо ваги деквантуються перед кожним множенням. Але це компенсується можливістю тренувати більшу модель на тому самому залізі.
- **Якість:** QLoRA дає **майже таку саму** якість (99.9% vs 100% для category, 97.8% vs 97.9% для intent). Різниця в межах статистичної похибки на 1000 тестових прикладах.

**Коли обрати QLoRA:**
1. **Мало пам'яті GPU** (T4, RTX 3060) - QLoRA дозволяє тренувати 4B-7B моделі.
2. **Велика модель** (13B+), яка не влізає в LoRA.
3. **Обмежений бюджет** (менше GPU-годин, але більше часу на тренування - компроміс).

**Коли обрати LoRA:**
1. **Достатньо пам'яті** (A100, RTX 4090 з 24 GB) - LoRA швидше.
2. **Мала модель** (0.6B, 1.7B) - QLoRA не дає великого виграшу.
3. **Коли час критичний** (тренування за 3.65 хв проти 4.95 хв).